In [1]:
"""
==========================================================
Feature Engineering
----------------------------------------------------------
Creates engineered features for the propensity model.

Input:
    outputs/eda_dataset.csv

Output:
    data/feature_engineered.csv
==========================================================
"""

'\n==========================================================\nFeature Engineering\n----------------------------------------------------------\nCreates engineered features for the propensity model.\n\nInput:\n    outputs/eda_dataset.csv\n\nOutput:\n    data/feature_engineered.csv\n==========================================================\n'

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
# ==========================================================
# Paths
# ==========================================================

BASE_DIR = Path.cwd().parent

INPUT_FILE = BASE_DIR / "outputs" / "eda_dataset.csv"
OUTPUT_FILE = BASE_DIR /  "data" / "feature_engineered.csv"

In [4]:
# ----------------------------------------------------------
# Load Dataset
# ----------------------------------------------------------

print("=" * 60)
print("Loading data...")
print("=" * 60)

df = pd.read_csv(INPUT_FILE)

df.shape

Loading data...


(500000, 19)

In [5]:
# ==========================================================
# Create Never Traded Flag
# ==========================================================

print("\nCreating never_traded feature...")

df["never_traded"] = (
    (
        (df["ord_cash"] == 0)
        &
        (df["ord_derv"] == 0)
        &
        (df["ord_comm"] == 0)
        &
        (df["ord_curr"] == 0)
    )
).astype(int)

df['never_traded'] = np.where(df["recency"].isna(),1,df['never_traded'])


Creating never_traded feature...


In [6]:
# ==========================================================
# Recency Treatment
# ==========================================================

print("Handling recency...")

# Non-traded customers get a very high recency
df.loc[df["never_traded"] == 1, "recency"] = 9999

# Any remaining missing values
df["recency"] = df["recency"].fillna(9999)

Handling recency...


In [7]:
# ==========================================================
# Log Transformations
# ==========================================================

print("Creating log transformed features...")

df["DP_Value_log"] = np.log1p(df["DP_Value"])

df["ord_cash_log"] = np.log1p(df["ord_cash"])

df["ord_derv_log"] = np.log1p(df["ord_derv"])

df["ord_comm_log"] = np.log1p(df["ord_comm"])

df["ord_curr_log"] = np.log1p(df["ord_curr"])

# Preserve negative margins
df["Total_Margin_log"] = np.sign(df["Total_Margin"]) * np.log1p(
    np.abs(df["Total_Margin"])
)

Creating log transformed features...


In [8]:
# ==========================================================
# Handle Missing Categoricals
# ==========================================================

print("Cleaning categorical columns...")

categorical_columns = [
    "self_dealer_status",
    "Vertical",
    "plan",
    "Category_Bucket_final",
    "Dealing_Zone"
]

for col in categorical_columns:

    df[col] = (
        df[col]
        .fillna("Missing")
        .astype(str)
        .str.strip()
    )

Cleaning categorical columns...


In [9]:
# ==========================================================
# Number_of_competition_account
# ==========================================================

print("Cleaning Number_of_competition_account...")

df["Number_of_competition_account"] = (
    df["Number_of_competition_account"]
    .fillna(0)
    .astype(int)
)

Cleaning Number_of_competition_account...


In [10]:
# ==========================================================
# Binary Variables
# ==========================================================

binary_cols = [
    "MTF_taken",
    "never_traded"
]

for col in binary_cols:
    df[col] = (
        df[col]
        .fillna(0)
        .astype(int)
    )

In [11]:
# ==========================================================
# Numeric Columns
# ==========================================================

numeric_cols = [
    "current_age",
    "entry_age",
    "vintage_months",
    "recency",
    "DP_Value_log",
    "Total_Margin_log",
    "ord_cash_log",
    "ord_derv_log",
    "ord_comm_log",
    "ord_curr_log",
    "Number_of_competition_account"
]

for col in numeric_cols:

    df[col] = pd.to_numeric(df[col], errors="coerce")

In [12]:
# ==========================================================
# Drop Raw Variables
# ==========================================================

print("Dropping raw variables...")

drop_columns = [
    "DP_Value",
    "Total_Margin",
    "ord_cash",
    "ord_derv",
    "ord_comm",
    "ord_curr"
]

existing = [c for c in drop_columns if c in df.columns]

df.drop(columns=existing, inplace=True)

Dropping raw variables...


In [13]:
# ==========================================================
# Summary
# ==========================================================

print("\nEngineered Dataset")

print(df.info())

print("\nMissing Values")

print(df.isnull().sum())


Engineered Dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 20 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Client_Code                    500000 non-null  object 
 1   current_age                    500000 non-null  int64  
 2   entry_age                      500000 non-null  int64  
 3   vintage_months                 500000 non-null  int64  
 4   Category_Bucket_final          500000 non-null  object 
 5   Vertical                       500000 non-null  object 
 6   self_dealer_status             500000 non-null  object 
 7   Number_of_competition_account  500000 non-null  int32  
 8   plan                           500000 non-null  object 
 9   Dealing_Zone                   500000 non-null  object 
 10  MTF_taken                      500000 non-null  int32  
 11  recency                        500000 non-null  float64
 12  target    

In [14]:
# ==========================================================
# Save
# ==========================================================

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_FILE, index=False)

print("\n" + "=" * 60)
print("Feature engineering completed successfully.")
print(f"Saved to:\n{OUTPUT_FILE}")
print("=" * 60)


Feature engineering completed successfully.
Saved to:
e:\github\AB Testing\data\feature_engineered.csv
